In [5]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
%matplotlib tk

import matplotlib.pyplot as plt
from src.viz import dibujarCampo, dibujarPuntosCenital
from src.CocoDataset import CocoDataset
from src.calibracion import marcarPuntos
from src.geometria import calcularHomografia, puntoApoyo, aMetros
from src.campo import separarPorCampo
from src.pipeline import analizarImagen

IMG_ID = 41

CORRESPONDENCIAS = [
    ((215, 228), (16.5, 13.84)),
    (( 30, 361), (16.5, 54.16)),
    ((498, 220), (52.5,  0.00)),
    ((499, 278), (52.5, 24.85)),
    ((504, 360), (52.5, 43.15)),
]

ds = CocoDataset("../data/raw/football-players/valid")

orden = ("corner_izq_lejano", "area_izq_lejana", "penalti_izq", "medio_lejano")

fig, ax = plt.subplots()
correspondencias = marcarPuntos(ds.imagen(IMG_ID), orden)

H_clics, Hinv = calcularHomografia(correspondencias)
H_mano, Hinv_mano = calcularHomografia(CORRESPONDENCIAS)

cajas = [c for c, k in ds.cajas(IMG_ID) if k == "player"]

puntos = []

for caja in cajas:
    puntos.append(puntoApoyo(caja))

metros = aMetros(H_clics, puntos)

dentro, fuera = separarPorCampo(metros)

pares = analizarImagen(ds, IMG_ID, cajas)
equipos = [e for _, e in pares]

m0 = [m for m, e in zip(metros, equipos) if e == "e0"]
m1 = [m for m, e in zip(metros, equipos) if e == "e1"]

fig, ax = plt.subplots(figsize=(15, 10))
dibujarCampo(ax)
dibujarPuntosCenital(ax, m0)
dibujarPuntosCenital(ax, m1, color="red")
plt.show()

fig.savefig("../outputs/cenital_41.png", dpi=120, bbox_inches="tight")


In [11]:
import numpy as np

np.linalg.norm(aMetros(H_clics, puntos) - aMetros(H_mano, puntos), axis=1)

x_clics = aMetros(H_clics, puntos)[:, 0]
x_mano  = aMetros(H_mano,  puntos)[:, 0]
d = x_clics - x_mano
print(np.abs(d[:, None] - d[None, :]).max())
print(np.round(d, 2))

for x, dd in sorted(zip(x_clics, d)):
    print(f"{x:6.1f}  {dd:6.2f}")

1.9008713
[ 0.26 -0.02  0.38  0.11  0.21 -0.06  0.07 -0.29  0.37  0.36  0.01  0.26
  0.39 -0.55 -0.56  0.05 -0.8  -1.51  0.35]
  20.8    0.38
  22.0    0.35
  23.1    0.26
  23.8    0.36
  23.9    0.21
  24.1    0.01
  24.2    0.26
  24.2    0.07
  24.2    0.11
  25.3   -0.06
  25.9   -0.02
  27.6    0.37
  27.8    0.05
  27.8    0.39
  41.9   -0.80
  48.8   -0.29
  50.7   -0.56
  53.6   -0.55
  53.7   -1.51
